# **N턴전 숫자 맞추기(Many-to-Many)**

* RNN은 이전 데이터를 기억할 수 있는 능력이 있습니다.
* 따라서 순서가 중요한 시퀀스데이터 처리에 사용되는 고전적인 모델입니다.
* 우리는 N턴전 숫자를 맞추는 RNN모델을 만들어보려고 합니다.

## **1. 데이터 준비하기**

- T,B,F (20, 32, 1) 크기의 랜덤 float형 데이터 (x)를 만들어 주세요.
- 데이터는 0부터 9사이의 숫자가 됩니다.
- 정답 (y)는 먼저 0으로 채워진 같은 크기의 데이터를 준비합니다.
- 그리고 y의 k번부터는 x의 앞에서부터 마지막 k개만 남긴 데이터로 덮어씌웁니다.

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# code
T = 20 # timestep의 수 (1번째 숫자, 2번째 숫자)
B = 32 # batch의 수    (한 batch = 32개의 숫자가 있음) 
K = 3  # k번째 전 숫자를 맞추는 목표

In [3]:
def make_data():
    X = torch.randint(0, 10, (T, B, 1)).float()  #randint(start, end, (shape))
    y = torch.zeros_like(X)
    # y = X의 K번째 전에 있는 숫자들이 답이 되어야 하는것.
    y[K:] = X[:-K]
    return X, y

In [4]:
# X.shape, y.shape

## **2. 모델 만들기**

레이어 구성은 다음과 같습니다.
- rnn layer -> Linear layer

In [5]:
# code
class RNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(1, 32)  # 1개 입력 -> 32개의 hidden공간으로 나눈다.(기억공간)
        self.linear = nn.Linear(32, 1)  # 기억 32개를 가지고 한개의 output을 낸다


    def forward(self, X):
        # self.rnn은 2개의 return (tuple)
        # h : 모든 시간 단계에 대한 출력
        # _ : 마지막 시점의 은닉 상태(마지막 cell)
        h, _ = self.rnn(X)
        return self.linear(h)
    # for _ in range() # <- 안 쓰는 변수들은 대부분 _ 로 표기한다.

In [6]:
model = RNN()
print(model)

RNN(
  (rnn): RNN(1, 32)
  (linear): Linear(in_features=32, out_features=1, bias=True)
)


## **3. 손실함수와 옵티마이저**

**Q1. 아래와 같이 옵티마이저를 설정합니다.**

- Optimizer : Adam
- learning rate : 0.003
- 손실 함수는 forward 내에서 F.mse_loss를 활용합니다.

In [7]:
# code
criterion = nn.MSELoss()

In [8]:
optimizer = torch.optim.Adam(model.parameters(), lr = 0.003)

## **4. 학습**

**Q1. 학습 모드로, 생성된 rnn 신경망을 학습시킵니다.**
- epochs:5000
- 1000번의 epoch마다 loss를 출력해보도록 합시다.


In [ ]:
# code
epochs = 5000

for epoch in range(epochs):
    model.train()
    # 순전파 -> loss -> 역전파(BPTT) -> update
    X, y = make_data()
    y_pred = model(X)       # 순전파
    loss = criterion(y_pred[K:], y[K:]) # loss + 1,2,3번쨰에는 답이 없는 0이니, loss를 구하는게 의미가 없음
    optimizer.zero_grad()
    loss.backward()         # 역전파(BPTT)
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch + 1} | loss: {loss}")


epoch 1000 | loss: 0.020446283742785454
epoch 2000 | loss: 0.012868636287748814
epoch 3000 | loss: 0.002536770887672901
epoch 4000 | loss: 0.0022743435110896826
epoch 5000 | loss: 0.0014271800173446536


## **5. 평가**

**Q1. 평가 모드로, 학습된 신경망의 정확도를 구합니다.**
- 총 20번 새로운 데이터를 만들어 예측을 시키고, 결과를 구해 봅시다.


In [ ]:
# model.eval()
# with torch.no_grad():
#     X, y = make_data()
#     y_pred = model(X)
# print(y_pred, y)

In [12]:
# code
test_epoch = 20
acc = 0 

for epoch in range(test_epoch):
    model.eval() 
    X, y = make_data()
    with torch.no_grad():
        # y_pred는 연속된 값이 나올테니 -> 근사치
        y_pred = model(X)
        # 근사치는 정수로 바꾸기 위해 반올림
        y_pred = y_pred.round()
        # 의미 없는 1,2,3번째 숫자 예측은 버리고, 나머지에 대해서 
        # 맞은거에 대한 평균만 내서 일단 합을 해준다.
        acc += (y_pred[K:] == y[K:]).float().mean().item()

# 20개의 test에 대한 평균을 내준다. -> accuracy
acc = acc / test_epoch
print(f"RNN의 accuracy: {acc}")

RNN의 accuracy: 1.0


## **6. N바꾸기**

**Q1. 5턴전 정확도를 평가해보자.**

In [13]:
# code
K = 5

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.003)

epochs = 5000

for epoch in range(epochs):
    model.train()
    # 순전파 -> loss -> 역전파(BPTT) -> update
    X, y = make_data()
    y_pred = model(X)       # 순전파
    loss = criterion(y_pred[K:], y[K:]) # loss + 1,2,3번쨰에는 답이 없는 0이니, loss를 구하는게 의미가 없음
    optimizer.zero_grad()
    loss.backward()         # 역전파(BPTT)
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch + 1} | loss: {loss}")


epoch 1000 | loss: 0.09102436155080795
epoch 2000 | loss: 0.016538506373763084
epoch 3000 | loss: 0.00902668759226799
epoch 4000 | loss: 0.014389642514288425
epoch 5000 | loss: 0.0055517954751849174


In [14]:
test_epoch = 20
acc = 0 

for epoch in range(test_epoch):
    model.eval() 
    X, y = make_data()
    with torch.no_grad():
        # y_pred는 연속된 값이 나올테니 -> 근사치
        y_pred = model(X)
        # 근사치는 정수로 바꾸기 위해 반올림
        y_pred = y_pred.round()
        # 의미 없는 1,2,3번째 숫자 예측은 버리고, 나머지에 대해서 
        # 맞은거에 대한 평균만 내서 일단 합을 해준다.
        acc += (y_pred[K:] == y[K:]).float().mean().item()

# 20개의 test에 대한 평균을 내준다. -> accuracy
acc = acc / test_epoch
print(f"RNN의 accuracy: {acc}")

RNN의 accuracy: 1.0


**Q2. 7턴전 정확도를 평가해보자.**

In [15]:
# code
K = 7

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.003)

epochs = 5000

for epoch in range(epochs):
    model.train()
    # 순전파 -> loss -> 역전파(BPTT) -> update
    X, y = make_data()
    y_pred = model(X)       # 순전파
    loss = criterion(y_pred[K:], y[K:]) # loss + 1,2,3번쨰에는 답이 없는 0이니, loss를 구하는게 의미가 없음
    optimizer.zero_grad()
    loss.backward()         # 역전파(BPTT)
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch + 1} | loss: {loss}")


epoch 1000 | loss: 8.15071964263916
epoch 2000 | loss: 0.08027581125497818
epoch 3000 | loss: 0.033658359199762344
epoch 4000 | loss: 0.04309270903468132
epoch 5000 | loss: 0.01329001598060131


In [16]:
# code
test_epoch = 20
acc = 0 

for epoch in range(test_epoch):
    model.eval() 
    X, y = make_data()
    with torch.no_grad():
        # y_pred는 연속된 값이 나올테니 -> 근사치
        y_pred = model(X)
        # 근사치는 정수로 바꾸기 위해 반올림
        y_pred = y_pred.round()
        # 의미 없는 1,2,3번째 숫자 예측은 버리고, 나머지에 대해서 
        # 맞은거에 대한 평균만 내서 일단 합을 해준다.
        acc += (y_pred[K:] == y[K:]).float().mean().item()

# 20개의 test에 대한 평균을 내준다. -> accuracy
acc = acc / test_epoch
print(f"RNN의 accuracy: {acc}")

RNN의 accuracy: 0.999158650636673


**Q3. 10턴전 정확도를 평가해보자.**

In [20]:
# code
K = 10

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.003)

epochs = 10000

for epoch in range(epochs):
    model.train()
    # 순전파 -> loss -> 역전파(BPTT) -> update
    X, y = make_data()
    y_pred = model(X)       # 순전파
    loss = criterion(y_pred[K:], y[K:]) # loss + 1,2,3번쨰에는 답이 없는 0이니, loss를 구하는게 의미가 없음
    optimizer.zero_grad()
    loss.backward()         # 역전파(BPTT)
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch + 1} | loss: {loss}")


epoch 1000 | loss: 8.765815734863281
epoch 2000 | loss: 7.9015793800354
epoch 3000 | loss: 7.188018798828125
epoch 4000 | loss: 8.794713973999023
epoch 5000 | loss: 2.0139214992523193
epoch 6000 | loss: 0.048790037631988525
epoch 7000 | loss: 0.0417255274951458
epoch 8000 | loss: 0.040315426886081696
epoch 9000 | loss: 0.01585741899907589
epoch 10000 | loss: 0.028866564854979515


In [21]:
# code
test_epoch = 20
acc = 0 

for epoch in range(test_epoch):
    model.eval() 
    X, y = make_data()
    with torch.no_grad():
        # y_pred는 연속된 값이 나올테니 -> 근사치
        y_pred = model(X)
        # 근사치는 정수로 바꾸기 위해 반올림
        y_pred = y_pred.round()
        # 의미 없는 1,2,3번째 숫자 예측은 버리고, 나머지에 대해서 
        # 맞은거에 대한 평균만 내서 일단 합을 해준다.
        acc += (y_pred[K:] == y[K:]).float().mean().item()

# 20개의 test에 대한 평균을 내준다. -> accuracy
acc = acc / test_epoch
print(f"RNN의 accuracy: {acc}")

RNN의 accuracy: 0.9973437428474426


**Q4. 15턴전 정확도를 평가해보자.**

In [23]:
# code
K = 15

model = RNN()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.003)

epochs = 10000

for epoch in range(epochs):
    model.train()
    # 순전파 -> loss -> 역전파(BPTT) -> update
    X, y = make_data()
    y_pred = model(X)       # 순전파
    loss = criterion(y_pred[K:], y[K:]) # loss + 1,2,3번쨰에는 답이 없는 0이니, loss를 구하는게 의미가 없음
    optimizer.zero_grad()
    loss.backward()         # 역전파(BPTT)
    optimizer.step()

    if (epoch + 1) % 1000 == 0:
        print(f"epoch {epoch + 1} | loss: {loss}")


epoch 1000 | loss: 9.196311950683594
epoch 2000 | loss: 8.505708694458008
epoch 3000 | loss: 6.998593807220459
epoch 4000 | loss: 8.912500381469727
epoch 5000 | loss: 8.583074569702148
epoch 6000 | loss: 7.6004767417907715
epoch 7000 | loss: 6.328851222991943
epoch 8000 | loss: 5.455432891845703
epoch 9000 | loss: 4.366447925567627
epoch 10000 | loss: 3.0311131477355957


In [24]:
# code
test_epoch = 20
acc = 0 

for epoch in range(test_epoch):
    model.eval() 
    X, y = make_data()
    with torch.no_grad():
        # y_pred는 연속된 값이 나올테니 -> 근사치
        y_pred = model(X)
        # 근사치는 정수로 바꾸기 위해 반올림
        y_pred = y_pred.round()
        # 의미 없는 1,2,3번째 숫자 예측은 버리고, 나머지에 대해서 
        # 맞은거에 대한 평균만 내서 일단 합을 해준다.
        acc += (y_pred[K:] == y[K:]).float().mean().item()

# 20개의 test에 대한 평균을 내준다. -> accuracy
acc = acc / test_epoch
print(f"RNN의 accuracy: {acc}")

RNN의 accuracy: 0.20125000104308127
